In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

In [ ]:
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "ml_df_v4.csv").exists() and (ROOT.parent / "data" / "ml_df_v4.csv").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

artifact_path = ROOT / "models" / "worldcup_model_v4.joblib"
artifact = joblib.load(artifact_path)

model = artifact["model"]
feature_cols = artifact["feature_cols"]
class_order = artifact["class_order"]
score_models = artifact.get("score_models", {})
primary_prediction = artifact.get("primary_prediction", "outcome")  # "score" = derive outcome from predicted scores

ml_df = pd.read_csv(ROOT / "data" / "ml_df_v4.csv")
if "date" in ml_df.columns:
    ml_df["date"] = pd.to_datetime(ml_df["date"], errors="coerce")

if "ranking_diff" in feature_cols and "ranking_diff" not in ml_df.columns:
    if "home_ranking_score" in ml_df.columns and "away_ranking_score" in ml_df.columns:
        ml_df["ranking_diff"] = ml_df["home_ranking_score"] - ml_df["away_ranking_score"]
    else:
        ml_df["ranking_diff"] = np.nan

print("Model: worldcup_model_v4")
print("Features:", len(feature_cols))
ml_df[["year", "home_team", "away_team", "home_wc_goals_before", "away_wc_goals_before", "result_target"]].head()

Model: worldcup_model_v4
Features: 161


,year,home_team,away_team,home_wc_goals_before,away_wc_goals_before,result_target
0,1930,France,Mexico,0,0,HomeWin
1,1930,United States,Belgium,0,0,HomeWin
2,1930,Romania,Peru,0,0,HomeWin
3,1930,Yugoslavia,Brazil,0,0,HomeWin
4,1930,Argentina,France,0,4,HomeWin


## Pick 2022 matches (2 per stage + Final)

Same selection as n5 for comparison.

In [3]:
ml_df_2022 = ml_df[ml_df["year"] == 2022].reset_index(drop=True)
stage_series = ml_df_2022["stage"].astype(str).str.lower()

def pick_stage(pattern, n):
    return ml_df_2022[stage_series.str.contains(pattern)].sort_values("date").head(n)

group_matches = pick_stage("group", 2)
r16_matches = pick_stage("round of 16", 2) if (stage_series.str.contains("round of 16")).any() else pick_stage("quarter", 2)
quarter_matches = pick_stage("quarter", 2) if (stage_series.str.contains("quarter")).any() else pd.DataFrame([])
semi_matches = pick_stage("semi", 2) if (stage_series.str.contains("semi")).any() else pd.DataFrame([])

final_mask = stage_series.eq("final")
final_match = (
    ml_df_2022[final_mask].sort_values("date").head(1)
    if final_mask.any()
    else ml_df_2022[stage_series.str.contains("final") & ~stage_series.str.contains("semi")].sort_values("date").head(1)
)

parts = [group_matches, r16_matches, quarter_matches, semi_matches, final_match]
test_rows = pd.concat([p for p in parts if len(p) > 0]).drop_duplicates().reset_index(drop=True)

test_rows[["date", "stage", "home_team", "away_team", "result_target"]]

,date,stage,home_team,away_team,result_target
0,2022-11-20,Group stage,Qatar,Ecuador,AwayWin
1,2022-11-21,Group stage,England,Iran,HomeWin
2,2022-12-03,round of 16,Argentina,Australia,HomeWin
3,2022-12-03,round of 16,Netherlands,United States,HomeWin
4,2022-12-09,quarter-finals,Croatia,Brazil,Draw
5,2022-12-09,quarter-finals,Netherlands,Argentina,Draw
6,2022-12-13,semi-finals,Argentina,Croatia,HomeWin
7,2022-12-14,semi-finals,France,Morocco,HomeWin
8,2022-12-18,final,Argentina,France,Draw


## Predict on selected matches (with Monte Carlo and score prediction)

Score-first: outcome derived from predicted score. Outcome model provides probabilities for Monte Carlo. Score models provide home/away goals for app.

In [4]:
results = []
score_home_model = score_models.get("home_goals")
score_away_model = score_models.get("away_goals")


def round_goal(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    return max(0, int(np.floor(float(x) + 0.5)))


def mc_simulate_outcomes(proba, class_order, n=10000, seed=42):
    """Monte Carlo: sample outcomes from probability distribution, return empirical probs."""
    rng = np.random.default_rng(seed)
    sims = rng.choice(class_order, size=n, p=proba)
    counts = pd.Series(sims).value_counts().reindex(class_order, fill_value=0)
    return (counts / n).to_dict()


def score_to_outcome(pred_h, pred_a):
    h, a = int(np.round(np.clip(pred_h, 0, 10))), int(np.round(np.clip(pred_a, 0, 10)))
    return "HomeWin" if h > a else ("AwayWin" if a > h else "Draw")

for i, row in test_rows.iterrows():
    X = pd.DataFrame([row]).reindex(columns=feature_cols)
    proba = model.predict_proba(X)[0]
    pred = class_order[int(np.argmax(proba))]
    actual = row["result_target"]

    pred_home_goals = pred_away_goals = None
    if score_home_model and score_away_model:
        pred_home_goals = round_goal(float(score_home_model.predict(X)[0]))
        pred_away_goals = round_goal(float(score_away_model.predict(X)[0]))
        if primary_prediction == "score" and pred_home_goals is not None and pred_away_goals is not None:
            pred = score_to_outcome(pred_home_goals, pred_away_goals)

    mc_probs = mc_simulate_outcomes(proba, class_order, n=20000, seed=42 + i)
    mc_pred = max(mc_probs, key=mc_probs.get)

    results.append({
        "date": row.get("date"),
        "stage": row.get("stage"),
        "home_team": row["home_team"],
        "away_team": row["away_team"],
        "actual": actual,
        "predicted": pred,
        "mc_predicted": mc_pred,
        "mc_prob_actual": mc_probs.get(actual, 0.0),
        "correct": pred == actual,
        "actual_home_goals": row.get("home_goals"),
        "actual_away_goals": row.get("away_goals"),
        "pred_home_goals": pred_home_goals,
        "pred_away_goals": pred_away_goals,
        "p_homewin": proba[class_order.index("HomeWin")],
        "p_draw": proba[class_order.index("Draw")],
        "p_awaywin": proba[class_order.index("AwayWin")],
        "mc_homewin": mc_probs.get("HomeWin", 0.0),
        "mc_draw": mc_probs.get("Draw", 0.0),
        "mc_awaywin": mc_probs.get("AwayWin", 0.0),
    })

results_df = pd.DataFrame(results)
results_df

,date,stage,home_team,away_team,actual,predicted,mc_predicted,mc_prob_actual,correct,actual_home_goals,actual_away_goals,pred_home_goals,pred_away_goals,p_homewin,p_draw,p_awaywin,mc_homewin,mc_draw,mc_awaywin
0,2022-11-20,Group stage,Qatar,Ecuador,AwayWin,AwayWin,HomeWin,0.00000,True,0.0,2.0,0,2,1.000000,0.000000,0.000000,1.00000,0.00000,0.00000
1,2022-11-21,Group stage,England,Iran,HomeWin,HomeWin,AwayWin,0.00000,True,6.0,2.0,5,2,0.000000,0.000000,1.000000,0.00000,0.00000,1.00000
2,2022-12-03,round of 16,Argentina,Australia,HomeWin,HomeWin,AwayWin,0.00000,True,2.0,1.0,2,1,0.000000,0.000000,1.000000,0.00000,0.00000,1.00000
3,2022-12-03,round of 16,Netherlands,United States,HomeWin,HomeWin,AwayWin,0.00000,True,3.0,1.0,2,1,0.000000,0.000000,1.000000,0.00000,0.00000,1.00000
4,2022-12-09,quarter-finals,Croatia,Brazil,Draw,AwayWin,HomeWin,0.00000,False,1.0,1.0,1,2,1.000000,0.000000,0.000000,1.00000,0.00000,0.00000
5,2022-12-09,quarter-finals,Netherlands,Argentina,Draw,HomeWin,AwayWin,0.49125,False,2.0,2.0,1,0,0.000000,0.500000,0.500000,0.00000,0.49125,0.50875
6,2022-12-13,semi-finals,Argentina,Croatia,HomeWin,Draw,AwayWin,0.00000,False,3.0,0.0,1,1,0.000000,0.142857,0.857143,0.00000,0.13960,0.86040
7,2022-12-14,semi-finals,France,Morocco,HomeWin,HomeWin,AwayWin,0.00000,True,2.0,0.0,2,1,0.000000,0.000000,1.000000,0.00000,0.00000,1.00000
8,2022-12-18,final,Argentina,France,Draw,AwayWin,HomeWin,0.28795,False,3.0,3.0,1,2,0.497175,0.282979,0.219846,0.49235,0.28795,0.21970


In [5]:
# Summary for selected matches (includes Monte Carlo metric)
summary = pd.DataFrame({
    "total_tests": [len(results_df)],
    "correct_predictions": [results_df["correct"].sum()],
    "accuracy": [results_df["correct"].mean()],
    "avg_mc_prob_actual": [results_df["mc_prob_actual"].mean()] if "mc_prob_actual" in results_df.columns else [None],
})
summary

,total_tests,correct_predictions,accuracy,avg_mc_prob_actual
0,9,5,0.555556,0.086578


## Full 2022 evaluation (all matches)

In [6]:
X_2022 = ml_df_2022.reindex(columns=feature_cols)
y_2022 = ml_df_2022["result_target"]

proba_2022 = model.predict_proba(X_2022)
pred_2022_outcome = [class_order[int(i)] for i in np.argmax(proba_2022, axis=1)]

# Score-first: derive outcome from predicted scores when score_models exist and primary_prediction=="score"
def score_to_outcome(pred_h, pred_a):
    h, a = int(np.round(np.clip(pred_h, 0, 10))), int(np.round(np.clip(pred_a, 0, 10)))
    return "HomeWin" if h > a else ("AwayWin" if a > h else "Draw")

use_score_derived = primary_prediction == "score" and score_models.get("home_goals") and score_models.get("away_goals")
if use_score_derived:
    pred_home_2022 = np.clip(score_models["home_goals"].predict(X_2022), 0, None)
    pred_away_2022 = np.clip(score_models["away_goals"].predict(X_2022), 0, None)
    pred_2022 = [score_to_outcome(h, a) for h, a in zip(pred_home_2022, pred_away_2022)]
    pred_score_2022 = [f"{int(np.round(h))}-{int(np.round(a))}" for h, a in zip(pred_home_2022, pred_away_2022)]
else:
    pred_2022 = pred_2022_outcome
    pred_score_2022 = [None] * len(y_2022)

full_results = pd.DataFrame({"actual": y_2022, "predicted": pred_2022})
if any(p is not None for p in pred_score_2022):
    full_results["pred_score"] = pred_score_2022
full_results["correct"] = full_results["actual"] == full_results["predicted"]
full_accuracy = full_results["correct"].mean()

if use_score_derived:
    print("Primary prediction: score-derived (win/lose from predicted score)")
print(f"Full 2022 accuracy: {full_accuracy:.2%}")
print(f"Correct: {full_results['correct'].sum()} / {len(full_results)}")

# Baseline: always HomeWin
baseline_pred = ["HomeWin"] * len(y_2022)
baseline_correct = (y_2022.values == np.array(baseline_pred))
baseline_accuracy = baseline_correct.mean()
print(f"Baseline (always HomeWin) accuracy: {baseline_accuracy:.2%}")

# Phase 5.5: Blend with prior when model underperforms baseline (45% HomeWin, 25% Draw, 30% AwayWin)
PRIOR_PROBA = np.array([0.45, 0.25, 0.30])
BLEND_WEIGHT = 0.3
proba_blended = (1 - BLEND_WEIGHT) * proba_2022 + BLEND_WEIGHT * PRIOR_PROBA
pred_blended = [class_order[int(i)] for i in np.argmax(proba_blended, axis=1)]
blend_accuracy = (y_2022.values == np.array(pred_blended)).mean()
if full_accuracy < baseline_accuracy and blend_accuracy > full_accuracy:
    print(f"Blended (70% model + 30% prior) accuracy: {blend_accuracy:.2%} (improvement)")

comparison_data = [
    ("worldcup_model_v4", full_accuracy, full_results["correct"].sum()),
    ("baseline_homewin", baseline_accuracy, baseline_correct.sum()),
]
if use_score_derived:
    outcome_acc = (y_2022.values == np.array(pred_2022_outcome)).mean()
    comparison_data.insert(1, ("v4_outcome_model", outcome_acc, (y_2022.values == np.array(pred_2022_outcome)).sum()))
if full_accuracy < baseline_accuracy and blend_accuracy > full_accuracy:
    comparison_data.append(("v4_blended_prior", blend_accuracy, (y_2022.values == np.array(pred_blended)).sum()))
comparison = pd.DataFrame(comparison_data, columns=["model", "accuracy", "correct_predictions"])
comparison["total_matches"] = len(y_2022)
comparison = comparison[["model", "accuracy", "total_matches", "correct_predictions"]]
comparison

Primary prediction: score-derived (win/lose from predicted score)
Full 2022 accuracy: 67.19%
Correct: 43 / 64
Baseline (always HomeWin) accuracy: 45.31%


,model,accuracy,total_matches,correct_predictions
0,worldcup_model_v4,0.671875,64,43
1,v4_outcome_model,0.218750,64,14
2,baseline_homewin,0.453125,64,29
